O serviço de vendas de carros usados Rusty Bargain está desenvolvendo um aplicativo para atrair novos clientes. Nesse aplicativo, você pode descobrir rapidamente o valor de mercado do seu carro. Você tem acesso a dados históricos: especificações técnicas, versões de acabamento e preços. Você precisa construir o modelo para determinar o valor. 

Rusty Bargain está interessado em:

- a qualidade da predição;
- a velocidade da predição;
- o tempo necessário para o treinamento

## Preparação de Dados

Importando as bibliotecas que serão utilizadas:

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
# Modelos de Referência
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
# Modelos de Gradient Boosting
from catboost import CatBoostRegressor
import lightgbm as lgb
# Otimizador de hiperparâmetros
from sklearn.model_selection import RandomizedSearchCV

Baixando os dados:

In [2]:
df = pd.read_csv('/datasets/car_data.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

Avaliando a integridade dos dados, vemos que há dados NULOS nas 5 seguintes características:

- VehicleType — tipo de carroçaria do veículo = 37.490 nulos (~10,6%)
- Gearbox — tipo de caixa de transmissão = 19.833 nulos (~5,6%)
- Model — modelo do veículo = 19.705 nulos (~5,6%)
- FuelType — tipo de combustível = 32.895 nulos (~9,3%)
- NotRepaired — veículo reparado ou não = **71.154 (~20,1%) ← maior atenção**

Isso já indica que dropar linhas pode custar caro (especialmente por causa de NotRepaired).

A investigação dos valores nulos começa pela avaliação do peso que esses dados faltantes podem estar exercendo em nossa base. Considerando que o modelo a ser construido deve determinar o valor ('Price'), iniciarei com um teste para avaliar como esse valor de 'Price' mediano se apresenta tanto na amostra com valores não-nulos, quanto na  amostra com valores apenas nulos.

In [3]:
# Loop de código para avaliar o preço mediano nas características que contém Nulos
for col in ['VehicleType','Gearbox','Model','FuelType','NotRepaired']:
    tmp = df.copy()
    tmp['is_na'] = tmp[col].isna()
    print(col, tmp.groupby('is_na')['Price'].median())

VehicleType is_na
False    2990
True     1199
Name: Price, dtype: int64
Gearbox is_na
False    2850
True     1000
Name: Price, dtype: int64
Model is_na
False    2800
True     1355
Name: Price, dtype: int64
FuelType is_na
False    2950
True     1100
Name: Price, dtype: int64
NotRepaired is_na
False    3200
True     1390
Name: Price, dtype: int64


Em todas as colunas com nulos, a mediana do preço é bem menor quando o valor está ausente:

- VehicleType: 2990 (não nulo) vs 1199 (nulo)
- Gearbox: 2850 vs 1000
- Model: 2800 vs 1355
- FuelType: 2950 vs 1100
- NotRepaired: 3200 vs 1390

Isso é um sinal de que esses valores estarem ausentes não é aleatório, e não devemos excluír esses dados. Para que nosso modelo possa interpretar esse sinal, podemos criar (1) um indicador de ausência (col_missing) e (2) substituir NaNs por uma categoria "unknown".

In [4]:
# Criando uma coluna com indicador de ausência e preenchendo NaNs com "Unknown"
cols_cat = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'NotRepaired']

for c in cols_cat:
    df[c + '_missing'] = df[c].isna().astype('int8')
    df[c] = df[c].fillna('unknown')

# SE OS VALORES das 'col_missing' esiverem zerados, reiniciar o Kernel e rodar o código uma única vez novamente
df.sample(10, random_state=1245)

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,...,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen,VehicleType_missing,Gearbox_missing,Model_missing,FuelType_missing,NotRepaired_missing
77252,26/03/2016 22:06,2200,sedan,2004,manual,103,astra,150000,9,petrol,...,no,26/03/2016 00:00,0,60388,27/03/2016 11:46,0,0,0,0,0
307423,11/03/2016 22:41,1700,sedan,2002,manual,82,a_klasse,150000,11,petrol,...,no,11/03/2016 00:00,0,89312,06/04/2016 11:17,0,0,0,0,0
342664,05/03/2016 16:41,1500,other,1999,auto,0,unknown,150000,1,petrol,...,yes,05/03/2016 00:00,0,66606,09/03/2016 11:44,0,0,1,0,0
221895,10/03/2016 02:36,2600,sedan,2004,manual,143,3er,150000,1,petrol,...,yes,10/03/2016 00:00,0,67433,10/03/2016 18:49,0,0,0,0,0
74012,27/03/2016 21:56,0,unknown,2000,manual,75,polo,150000,0,gasoline,...,unknown,27/03/2016 00:00,0,28277,05/04/2016 21:48,1,0,0,0,1
99504,11/03/2016 12:45,500,unknown,2017,unknown,41,cuore,150000,8,petrol,...,no,11/03/2016 00:00,0,38108,12/03/2016 01:45,1,1,0,0,0
169728,01/04/2016 20:57,1000,unknown,2016,unknown,0,3er,150000,2,petrol,...,unknown,01/04/2016 00:00,0,92637,01/04/2016 21:40,1,1,0,0,1
164031,21/03/2016 12:41,2100,small,2006,manual,97,rio,150000,1,petrol,...,no,21/03/2016 00:00,0,10589,26/03/2016 17:47,0,0,0,0,0
162146,25/03/2016 21:38,4400,convertible,2006,unknown,61,fortwo,80000,6,petrol,...,no,25/03/2016 00:00,0,66740,01/04/2016 09:15,0,1,0,0,0
127769,19/03/2016 23:57,2300,sedan,2000,manual,160,c_klasse,150000,5,petrol,...,no,19/03/2016 00:00,0,40231,20/03/2016 07:46,0,0,0,0,0


Agora podemos entender outros erros que podem estar afetando nossos dados, a começar pela potência "Power".

In [5]:
# Observando valores anunciados de potência
df['Power'].value_counts()

0        40225
75       24023
60       15897
150      14590
101      13298
         ...  
323          1
3454         1
1056         1
13636        1
1158         1
Name: Power, Length: 712, dtype: int64

Vemos muitos valores zerados de potência (Power == 0), assim como muitos valores outliers (Power = 3454, 1056, 13636, 1158) fora do domínio de potência de carros (HP), então são erros/ruídos. Vamos observar mais de perto o impacto que esses ZEROS podem estar tendo em nossos dados, pois - em anúncio de carro - esses valores podem ser um *missing* disfarçado.

In [6]:
# Qual o percentual de ZEROS em nossa base
zero_rate = (df['Power'] == 0).mean()
print(f"Cerca de {zero_rate:.2%} dos valores registrados em Power são 0.")

Cerca de 11.35% dos valores registrados em Power são 0.


Cerca de 11.3% dos valores de Power são 0. Isso indica que provavelmente esses registros são NaNs escondidos, que devemos tratar. Mas antes de solucionarmos, vamos avaliar o topo da nossa amostra.

In [7]:
df['Power'].describe(percentiles=[.5, .9, .95, .99, .995, .999])

count    354369.000000
mean        110.094337
std         189.850405
min           0.000000
50%         105.000000
90%         179.000000
95%         218.000000
99%         300.000000
99.5%       334.000000
99.9%       700.000000
max       20000.000000
Name: Power, dtype: float64

A análise acima nos permite ver que nessa base de carros usados, valores acima de 700 hp são extremamente raros (≈0,1%) e o topo absoluto (20.000) é ruído. Para tratar os zeros os outliers absurdos de nossa base em Power, vamos usar a seguinte função:

In [8]:
# Função para limpar os valores em Power
def clean_power(df, col='Power', max_hp=700, create_missing_flag=True):
    """
    Trata missing disfarçado e outliers no topo para a coluna de potência.
    
    Regras:
      - Power == 0  -> NaN (missing disfarçado)
      - Power > max_hp -> NaN (corte do topo)
      - (opcional) cria flag Power_missing antes de imputar
    
    Retorna:
      df (com Power limpo) e um dicionário com estatísticas do que foi alterado.
    """
    df = df.copy()

    # Garantir tipo numérico (se vier como string em alguma etapa)
    df[col] = pd.to_numeric(df[col], errors='coerce')

    # Contagens antes
    n_total = len(df)
    n_zero_before = (df[col] == 0).sum()
    n_over_before = (df[col] > max_hp).sum()
    n_na_before = df[col].isna().sum()

    # Missing disfarçado e corte do topo
    df.loc[df[col] == 0, col] = np.nan
    df.loc[df[col] > max_hp, col] = np.nan

    # Flag de missing (recomendado para o modelo)
    if create_missing_flag:
        df[f'{col}_missing'] = df[col].isna().astype('int8')

    # Contagens depois
    n_na_after = df[col].isna().sum()

    # Preenchendo os NaNs
    median_power = df['Power'].median()
    df['Power'] = df['Power'].fillna(median_power)

    stats = {
        "linhas_total": n_total,
        "zeros_convertidos_para_NA": int(n_zero_before),
        "acima_do_limite_convertidos_para_NA": int(n_over_before),
        "NA_antes": int(n_na_before),
        "NA_depois": int(n_na_after),
        "NA_adicionados": int(n_na_after - n_na_before),
        "limite_max_hp": max_hp
    }

    return df, stats

In [9]:
# Aplicando a função
df, power_stats = clean_power(df, col='Power', max_hp=700, create_missing_flag=True)
print(power_stats)

{'linhas_total': 354369, 'zeros_convertidos_para_NA': 40225, 'acima_do_limite_convertidos_para_NA': 354, 'NA_antes': 0, 'NA_depois': 40579, 'NA_adicionados': 40579, 'limite_max_hp': 700}


In [10]:
# Amostragem com Power corrigido.
df.sample(10, random_state=1245)

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,...,DateCreated,NumberOfPictures,PostalCode,LastSeen,VehicleType_missing,Gearbox_missing,Model_missing,FuelType_missing,NotRepaired_missing,Power_missing
77252,26/03/2016 22:06,2200,sedan,2004,manual,103.0,astra,150000,9,petrol,...,26/03/2016 00:00,0,60388,27/03/2016 11:46,0,0,0,0,0,0
307423,11/03/2016 22:41,1700,sedan,2002,manual,82.0,a_klasse,150000,11,petrol,...,11/03/2016 00:00,0,89312,06/04/2016 11:17,0,0,0,0,0,0
342664,05/03/2016 16:41,1500,other,1999,auto,110.0,unknown,150000,1,petrol,...,05/03/2016 00:00,0,66606,09/03/2016 11:44,0,0,1,0,0,1
221895,10/03/2016 02:36,2600,sedan,2004,manual,143.0,3er,150000,1,petrol,...,10/03/2016 00:00,0,67433,10/03/2016 18:49,0,0,0,0,0,0
74012,27/03/2016 21:56,0,unknown,2000,manual,75.0,polo,150000,0,gasoline,...,27/03/2016 00:00,0,28277,05/04/2016 21:48,1,0,0,0,1,0
99504,11/03/2016 12:45,500,unknown,2017,unknown,41.0,cuore,150000,8,petrol,...,11/03/2016 00:00,0,38108,12/03/2016 01:45,1,1,0,0,0,0
169728,01/04/2016 20:57,1000,unknown,2016,unknown,110.0,3er,150000,2,petrol,...,01/04/2016 00:00,0,92637,01/04/2016 21:40,1,1,0,0,1,1
164031,21/03/2016 12:41,2100,small,2006,manual,97.0,rio,150000,1,petrol,...,21/03/2016 00:00,0,10589,26/03/2016 17:47,0,0,0,0,0,0
162146,25/03/2016 21:38,4400,convertible,2006,unknown,61.0,fortwo,80000,6,petrol,...,25/03/2016 00:00,0,66740,01/04/2016 09:15,0,1,0,0,0,0
127769,19/03/2016 23:57,2300,sedan,2000,manual,160.0,c_klasse,150000,5,petrol,...,19/03/2016 00:00,0,40231,20/03/2016 07:46,0,0,0,0,0,0


Agora vamos investigar RegistrationYear...

In [11]:
# Verificando a data mais recente em que um perfil foi baixado do BD (REMOVER '#' ABAIXO PARA TESTAR)
# print("Latest DateCrawled (UTC):", pd.to_datetime(df['DateCrawled'], errors='coerce', utc=True).max())

A data mais recente em que um perfil foi baixado do BD foi 2016-12-03 23:59:00+00:00

In [12]:
# Cálculodos valores permitidos em nossa base.
percentual_invalidos = (
    (df['RegistrationYear'] < 1886) | # O ano de 1886 é considerado o ano do primeiro automóvel (Karl Benz).
    (df['RegistrationYear'] > 2016)
).mean() * 100

print(percentual_invalidos)

4.118870442956354


Cerca de 4,12% dos nossos registros de RegistrationYear estão impróprios para uso. Vamos verificar se há alguma discrepância de preço entre esses casos e os casos válidos.

In [13]:
# Verificando se valores inválidos estão concentrados em alguma faixa de preço
df.loc[
    (df['RegistrationYear'] < 1886) | 
    (df['RegistrationYear'] > 2016),
    'Price'
].describe()

count    14596.000000
mean      3144.472801
std       3459.888043
min          0.000000
25%        950.000000
50%       1899.000000
75%       3999.000000
max      20000.000000
Name: Price, dtype: float64

Os registros com ano inválido não são outliers de preço. Eles seguem um padrão normal da base (carros mais baratos, provavelmente mais antigos).

Isso sugere que:
- Não parece ser fraude ou erro sistemático grave
- Provavelmente são erros de digitação ou dados ausentes mal preenchidos

Para tratar desse erro, usaremos de duas estratégias:
1) Vamos criar uma nova coluna 'RegistrationYear_missing' para informar nosso modelo dessas flags de valores inválidos; e
2) Vamos substituir esses valores errôneos pela mediana. No entanto, vamos levar em consideração algumas variáveis, como Model e Brand, para tentar reduzir distorções de sinal. Afinal, um imputação por grupo mantém a variabilidade natural entre segmentos.

In [14]:
# Criando flag de 'RegistrationYear_missing'
df['RegistrationYear_missing'] = (
    (df['RegistrationYear'] < 1886) |
    (df['RegistrationYear'] > 2016)
).astype(int)

In [15]:
# Substituindo valores inválidos por NaN
df.loc[df['RegistrationYear_missing'] == 1, 'RegistrationYear'] = np.nan

Agora vamos substituir os novos NaNs pela mediana seguindo alguns critérios de agrupamento. Esses critérios foram estabelecidos para os casos extremos em que um tipo de agrupamento (ex. por Model) não funcione devido à amostragem insuficiente de veículos para fornecer a mediana.

Nesse caso, o valor permaneceria NaN, e o próximo critério de agrupamento tentaria preenchê-lo.

In [16]:
# 1 Mediana por modelo
df['RegistrationYear'] = (
    df.groupby('Model')['RegistrationYear']
      .transform(lambda x: x.fillna(x.median()))
)

# 2 Mediana por marca (Brand)
df['RegistrationYear'] = (
    df.groupby('Brand')['RegistrationYear']
      .transform(lambda x: x.fillna(x.median()))
)

# 3 Mediana global
df['RegistrationYear'].fillna(df['RegistrationYear'].median(), inplace=True)

Agora, vamos checar o RegistrationMonth...

In [17]:
df['RegistrationMonth'].value_counts()

0     37352
3     34373
6     31508
4     29270
5     29153
7     27213
10    26099
12    24289
11    24186
9     23813
1     23219
8     22627
2     21267
Name: RegistrationMonth, dtype: int64

Aqui vemos que o maior volume de RegistrationMonth é 0, ou seja, um NaN disfarçado. Sendo assim, vamos criar uma flag 'RegistrationMonth_missing', e imputar o valor mediano global. Neste caso, o mês não deve exercer um impacto pequeno na previsão de preço, então não precisamos usar critérios mais complexos de agrupamento considerando outras variáveis.

In [18]:
# Transformando valores 0 em NaNs
df.loc[df['RegistrationMonth'] == 0, 'RegistrationMonth'] = np.nan

# Criando flag de missing
df['RegistrationMonth_missing'] = df['RegistrationMonth'].isna().astype(int)

#Preenhendo valor NaN com mediana
df['RegistrationMonth'].fillna(df['RegistrationMonth'].median(), inplace=True)

In [19]:
# Revertendo as colunas Registration para o formato int
df['Power'] = df['Power'].astype(int)
df['RegistrationYear'] = df['RegistrationYear'].astype(int)
df['RegistrationMonth'] = df['RegistrationMonth'].astype(int)

Agora, finalmente, vamos avaliar Price. Como ela é a variável alvo do modelo, é importante tratar e analisar ela corretamente.

In [20]:
# Quantos preços zerados?
(df['Price'] == 0).sum()

10772

In [21]:
df['Price'].describe(percentiles=[.03, .1, .25, .5, .9, .95, .99, .995, .999])

count    354369.000000
mean       4416.656776
std        4514.158514
min           0.000000
3%            0.000000
10%         499.000000
25%        1050.000000
50%        2700.000000
90%       11450.000000
95%       14600.000000
99%       18800.000000
99.5%     19500.000000
99.9%     19999.000000
max       20000.000000
Name: Price, dtype: float64

Com os testes àcima, vemos que cerca de 3% do valores Price de nossa base estão zerados (10.772/354.369); e a maior parte da base está entre 500 e 14.600, com um pequeno volume de outliers extremos, tanto para preços muito altos, como para preços muito baixos.

Com isso, vamos...
- Remover os registros com Price == 0; e
- Filtrar os outliers mais extremos: àcima de 99,5%.

In [22]:
# Removendo os registros com Price == 0
df = df[df['Price'] > 0]

In [23]:
# Limitando os registros até 99,5º percentil
upper_limit = df['Price'].quantile(0.995)
df = df[df['Price'] <= upper_limit]

Como nosso modelo não consegue trabalhar com formatos de data, vamos remover essas características de nossos dados.

In [24]:
df.drop(['DateCrawled', 'DateCreated', 'LastSeen'], axis=1, inplace=True)

## Treinamento do modelo

Separando amostras de teste:

In [25]:
features_train, features_valid, target_train, target_valid = train_test_split(
    df.drop('Price', axis=1), df.Price, test_size=0.25, random_state=12345
)

Selecionando features a serem testadas:

In [26]:
features = [
    'VehicleType',
    'RegistrationYear',
    'Gearbox',
    'Power',
    'Model',
    'Mileage',
    'RegistrationMonth',
    'FuelType',
    'Brand',
    'NotRepaired',
    'VehicleType_missing',
    'Gearbox_missing',
    'Model_missing',
    'FuelType_missing',
    'NotRepaired_missing',
    'Power_missing',
    'RegistrationYear_missing',
    'RegistrationMonth_missing',
]

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 341981 entries, 0 to 354368
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Price                      341981 non-null  int64 
 1   VehicleType                341981 non-null  object
 2   RegistrationYear           341981 non-null  int64 
 3   Gearbox                    341981 non-null  object
 4   Power                      341981 non-null  int64 
 5   Model                      341981 non-null  object
 6   Mileage                    341981 non-null  int64 
 7   RegistrationMonth          341981 non-null  int64 
 8   FuelType                   341981 non-null  object
 9   Brand                      341981 non-null  object
 10  NotRepaired                341981 non-null  object
 11  NumberOfPictures           341981 non-null  int64 
 12  PostalCode                 341981 non-null  int64 
 13  VehicleType_missing        341981 non-null  

### Modelos Convencionais: __Regressão Linearm, Árvore de Decisão e Floresta Aleatória__

In [28]:
# Para Regressão Linear , é necessário usar One-Hot Encoding
categorical_features = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']

# Cria colunas binárias para cada categoria
X_train_enc = pd.get_dummies(features_train, columns=categorical_features, drop_first=True)
X_test_enc = pd.get_dummies(features_valid, columns=categorical_features, drop_first=True)

# Alinhar colunas (garante que treino e teste tenham as mesmas colunas)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

linear_model = LinearRegression()

linear_model.fit(X_train_enc, target_train)

LinearRegression()

In [29]:
# Criar o modelo de Árvore de Decisão
tree_model = DecisionTreeRegressor()

# Definição de hiperparâmetros para otimizar o modelo
param_dist = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': [None, 'sqrt', 'log2']
}

# Otimizando hiperparâmetros
tree_search = RandomizedSearchCV(estimator=tree_model, param_distributions=param_dist,
n_iter=25, cv=5, random_state=12345, n_jobs=-1)

# Treinar
tree_search.fit(X_train_enc, target_train)

# Melhores hiperparâmetros
print("Melhores parâmetros:", tree_search.best_params_)

Melhores parâmetros: {'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': None, 'max_depth': 20}


Os melhores hiperparâmetros para Árvore de Decisão são...
{'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': None, 'max_depth': 20}

In [30]:
# Criar o modelo Floresta Aleatória
rf_model = RandomForestRegressor()

# Definição de hiperparâmetros para otimizar modelo
param_grid = {
'n_estimators': [5, 10, 20],
'max_features': ['auto', 'sqrt', 'log2'],
'max_depth': [5, 10, 20],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'bootstrap': [True, False]
}

# Otimizando hiperparâmetros
rf_search = RandomizedSearchCV(estimator=rf_model, param_distributions=param_grid,
n_iter=25, cv=2, random_state=12345, n_jobs=-1)

# Treinar
rf_search.fit(X_train_enc, target_train)

# Melhores hiperparâmetros
print("Melhores parâmetros:", rf_search.best_params_)

Melhores parâmetros: {'n_estimators': 5, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'auto', 'max_depth': 20, 'bootstrap': True}


Os melhores hiperparâmetros para Floresta Aleatória são... {'n_estimators': 5, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'auto', 'max_depth': 20, 'bootstrap': True}

### Modelo __CatBoost__:

Treinando o modelo Catboost:

In [31]:
catboost_model = CatBoostRegressor(loss_function="RMSE", iterations=1000, depth=6, learning_rate=0.2, random_seed=12345)

catboost_model.fit(features_train, target_train, cat_features=features, verbose=100)

0:	learn: 3851.0079772	total: 264ms	remaining: 4m 23s
100:	learn: 1787.9643605	total: 27.8s	remaining: 4m 7s
200:	learn: 1713.5116067	total: 56.6s	remaining: 3m 44s
300:	learn: 1672.6197471	total: 1m 25s	remaining: 3m 19s
400:	learn: 1643.0419645	total: 1m 55s	remaining: 2m 52s
500:	learn: 1621.6722337	total: 2m 25s	remaining: 2m 25s
600:	learn: 1603.7539127	total: 2m 55s	remaining: 1m 56s
700:	learn: 1588.0253610	total: 3m 26s	remaining: 1m 27s
800:	learn: 1573.0826187	total: 3m 57s	remaining: 58.9s
900:	learn: 1558.8958866	total: 4m 27s	remaining: 29.4s
999:	learn: 1546.9739502	total: 4m 58s	remaining: 0us


### Modelo __LightGBM__:

In [32]:
categorical_features = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']

for col in categorical_features:
    features_train[col] = features_train[col].astype('category')
    features_valid[col] = features_valid[col].astype('category')

# Conjuntos de treino e teste
train_data = lgb.Dataset(features_train, label=target_train)
test_data = lgb.Dataset(features_valid, label=target_valid, reference=train_data)

# Parâmetros básicos
params = {
    'objective': 'regression',   # regressão
    'metric': 'rmse',            # métrica de avaliação
    'learning_rate': 0.1,
    'num_leaves': 31,
    'max_depth': -1,             # sem limite de profundidade
    'random_state': 123
}

# Treinar
lgb_model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,          # número de árvores
    valid_sets=[test_data],
    early_stopping_rounds=50,     # para parar se não melhorar
    verbose_eval=100
)

/.venv/lib/python3.9/site-packages/lightgbm/engine.py:181: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
/.venv/lib/python3.9/site-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015773 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 927
[LightGBM] [Info] Number of data points in the train set: 256485, number of used features: 19
[LightGBM] [Info] Start training from score 4480.954933
Training until validation scores don't improve for 50 rounds


/.venv/lib/python3.9/site-packages/lightgbm/basic.py:1780: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning('Overriding the parameters from Reference Dataset.')
/.venv/lib/python3.9/site-packages/lightgbm/basic.py:1513: UserWarning: categorical_column in param dict is overridden.
  _log_warning(f'{cat_alias} in param dict is overridden.')


[100]	valid_0's rmse: 1656.71
[200]	valid_0's rmse: 1621.3
[300]	valid_0's rmse: 1606.26
[400]	valid_0's rmse: 1594.09
[500]	valid_0's rmse: 1585.82
[600]	valid_0's rmse: 1579.73
[700]	valid_0's rmse: 1575.06
[800]	valid_0's rmse: 1569.93
[900]	valid_0's rmse: 1565.46
[1000]	valid_0's rmse: 1562.65
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 1562.65


## Análise do modelo

Predisendo com os modelos convencionais:

In [33]:
linear_pred = linear_model.predict(X_test_enc)
tree_pred = tree_search.predict(X_test_enc)
rf_pred = rf_search.predict(X_test_enc)

Predisendo com o modelo Catboost:

In [34]:
catboost_pred = catboost_model.predict(features_valid)

Predisendo com o modelo LightGBM:

In [35]:
lgb_pred = lgb_model.predict(features_valid, num_iteration=lgb_model.best_iteration)

Avaliando predições:

In [36]:
predictions = [
    ("LinearRegression", linear_pred),
    ("DecisionTree", tree_pred),
    ("RandomForest", rf_pred),
    ("CatBoost", catboost_pred),
    ("LightGBM", lgb_pred)
]

for name, pred in predictions:
    rmse = np.sqrt(mean_squared_error(target_valid, pred))
    r2 = r2_score(target_valid, pred)
    print(f"Modelo: {name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.3f}")
    print()

Modelo: LinearRegression
RMSE: 2618.26
R²: 0.646

Modelo: DecisionTree
RMSE: 1853.43
R²: 0.823

Modelo: RandomForest
RMSE: 1721.57
R²: 0.847

Modelo: CatBoost
RMSE: 1625.31
R²: 0.864

Modelo: LightGBM
RMSE: 1562.65
R²: 0.874



Os modelos testados apresentaram desempenho crescente à medida que a complexidade e a capacidade de aprendizado aumentaram. Os resultados obtidos no conjunto de validação foram:

| Modelo               | RMSE     | R²    |
|---------------------|----------|-------|
| LinearRegression    | 2618.26  | 0.646 |
| DecisionTree        | 1853.43  | 0.823 |
| RandomForest        | 1721.57  | 0.847 |
| CatBoost            | 1630.24  | 0.863 |
| LightGBM            | 1562.65  | 0.874 |

### Observações:

1. **Qualidade da predição:**  
   - A qualidade melhora progressivamente do LinearRegression para modelos de boosting.  
   - **LightGBM** apresentou o melhor desempenho, com RMSE de 1562,65 e R² de 0,874, indicando que consegue explicar 87,4% da variação nos preços.  
   - Modelos lineares ou árvores individuais têm limitações em capturar não-linearidades e interações complexas, resultando em R² mais baixo.

2. **Velocidade da predição:**  
   - Modelos simples como **LinearRegression** são extremamente rápidos para predizer novos registros.
   - **DecisionTree**, **RandomForest**, **CatBoost** e **LightGBM** demoram mais, mas a diferença ainda é pequena para uso em tempo real, especialmente LightGBM, que é otimizado para predição rápida.
   - No caso de **DecisionTree** e **RandomForest** ainda há um agravante que reduz a velocidade aninhado na otimização de hiperparâmetros, no caso com uso de **RandomizedSearchCV**. Com poder e tempo de processamento o suficiente, é possível que esses modelos pudessem alcançar a precisão como a de Gradient Boosters, como os utilizados aqui, mas isso não seria a situação ideal.

3. **Tempo de treinamento:**  
   - LinearRegression treinam em frações de segundo.  
   - DecisionTree e RandomForest levam mais tempo devido à otimização de hiperparâmetros.  
   - CatBoost também exige mais tempo de treinamento (4min 58s) por causa do gradient boosting, mas esse custo é compensado pela alta precisão.
   - LightGBM costuma ser mais rápido que CatBoost em datasets grandes. No caso estudado, a velocidade de treinamento chegou a superar a dos modelos de DecisionTree e RandomForest.

### Conclusão:

- Para **alta precisão**, os modelos de **gradient boosting** (LightGBM e CatBoost) são claramente os mais indicados. Embora os modelos de Árvore de Decisão e Floresta Aleatória tenham alcançado desempenhos altos também.  
- Se a prioridade for **rapidez de predição**, LinearRegression ou LightGBM podem ser usados para estimativas mais rápidas, respectivamente. No caso de LinearRegression, com menor acurácia  
- Para **um bom compromisso entre precisão e velocidade**, **LightGBM** se destaca, oferecendo **melhor RMSE, R² alto e predição rápida**, sendo o mais adequado para o aplicativo da Rusty Bargain.